# 02 — TF-IDF

First "text → numbers" step. We take the `body` column from the corpus table
and turn it into a **document-term matrix (DTM)**: one row per article, one
column per word in the vocabulary, cells = TF-IDF weights.

Reminder of the two tables (they are different things):
- corpus table  → readable, one row per article, `body` is text  (notebook 01)
- DTM           → not readable, one row per article, columns are words, cells
                  are numbers  (this notebook)

The DTM is what LDA and similarity search will consume next.


In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_parquet("../data/corpus.parquet")
df.shape  # 81 rows

(81, 6)

In [2]:
df.head()

,source_file,page_number,era_label,headline,body,n_words
0,Pages 30-37_p01.json,30,1911-12,Battling Barnsley show Yorkshire grit,BARNSLEY played six goalless games in an extra...,255
1,Pages 30-37_p01.json,30,1911-12,Another Olympic triumph for England,ENGLAND represented the United Kingdom again a...,238
2,Pages 30-37_p02.json,31,1912-13,Battle of the giants ends level,THE BATTLE of the giants has ended with honour...,521
3,Pages 30-37_p02.json,31,1912-13,Ten Irish heroes,ENGLAND LOST to Ireland for the first time in ...,143
4,Pages 30-37_p03.json,32,1913-14,A great day for the Irish,IRELAND have won the International Championshi...,304


## Fit the vectorizer

`TfidfVectorizer` does three things in one call: tokenizes the text, builds the
vocabulary, and computes TF-IDF weights. For a first pass, keep it simple —
lowercase + English stopwords + drop words that appear in only one document.

You choose the knobs. Start with:
- `stop_words="english"`  — drop "the", "and", "of" …
- `min_df=2`              — ignore words appearing in <2 documents (kills typos
                            and one-off names that can't help compare articles)

Then `.fit_transform(df["body"])` returns the DTM as a sparse matrix `X`.


In [3]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_df=0.9,
    min_df=2,
)
X = vectorizer.fit_transform(df["body"])
X.shape   # (81, vocab_size)  -> how big is the vocabulary?

(81, 1488)

## Look at it

A DTM is unreadable in the raw, but two views make it concrete.

**1. The vocabulary.** `vectorizer.get_feature_names_out()` is the list of words
that became columns. How many? Peek at a slice.

**2. Top terms for one article.** Pick a row, pull its weights, sort descending,
and print the highest-weighted words. Do they look like what that article is
*about*? That's the whole intuition behind TF-IDF — the words that are frequent
*here* but rare *elsewhere* float to the top.


In [4]:
# TODO 1: how big is the vocabulary?
terms = vectorizer.get_feature_names_out()
len(terms)

1488

In [8]:
# TODO 2: top terms for one article.
#   - pick a row index i (try the Barnsley one, i=0)
#   - print df.loc[i, "headline"] so you know what you're looking at
#   - row = X[i].toarray().ravel()          # that doc's weights, dense
#   - take the indices of the largest weights (numpy argsort) and map back
#     through `terms` to words
#   Do the top words match the headline?
i=0
terms[i] # 'barnsley' is the first term in the vocabulary
print(df.loc[i, "headline"])
row = X[i].toarray().ravel() 
top_idx = np.argsort(row)[::-1][:10]  # positions of the 10 biggest weights
for j in top_idx:
    print(f"{terms[j]:20s} {row[j]:.3f}")

Battling Barnsley show Yorkshire grit
barnsley             0.524
albion               0.269
extra time           0.202
extra                0.193
time                 0.170
utley                0.160
attack               0.150
broke                0.141
goal                 0.134
past                 0.124


ngram_range=(1, 2) means "count both unigrams (single words) and bigrams (adjacent word pairs)," so extra, time, and extra time all become separate columns in the matrix.

## Where this goes next

`X` (the DTM) plus `terms` (the vocabulary) are the handoff to notebook 03 (LDA).
No need to save the matrix — it's cheap to rebuild, and we may change the knobs.

**Open question to sit with:** we used sklearn's built-in tokenizer here. It
doesn't lemmatize — "goal", "goals", "scored", "scoring" are all separate
columns. Does that matter for this corpus? Hold the thought; that's the spaCy
preprocessing step, and we'll decide whether it earns its place by comparing.
